In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import scipy as sp

pv.set_jupyter_backend('trame')


In [ ]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)



In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"

In [ ]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

In [ ]:
centers = hdf5_file["bc_func/centers"][:].astype(float)

In [ ]:
nk = len(hdf5_file["bc_func"].keys())-2
print(f"Number of keys in the HDF5 file: {nk}")

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

In [ ]:
# m=to_pyvista_mesh(V, T)
# # m=m.explode(0.5)
# plt = pv.Plotter()
# plt.add_mesh(m, show_edges=True, style='wireframe', line_width=1.0)
# plt.add_mesh(to_pyvista_mesh(V[top]), color='red', point_size=10, render_points_as_spheres=True, name='top')
# plt.add_mesh(to_pyvista_mesh(V[bottom]), color='green', point_size=10, render_points_as_spheres=True, name='bottom')
# plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=10, render_points_as_spheres=True, name='middle')
# plt.show()

In [ ]:
# m=m.explode(0.5)
plt = pv.Plotter()
plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=8, render_points_as_spheres=True, name='middle')
plt.add_mesh(to_pyvista_mesh(centers), color='red', point_size=7, render_points_as_spheres=True, name='middle')
plt.show()

In [ ]:
actor = None

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



In [ ]:
def icp(centers, d):
    p0 = centers
    p1 = centers + d
    mu0 = np.mean(p0, axis=0)
    mu1 = np.mean(p1, axis=0)

    p0 -= mu0
    p1 -= mu1
    t = mu0 - mu1

    cov = p1.T @ p0
    U, s, Ut = sp.linalg.svd(cov)
    R = Ut @ U.T

    return R, t

def align(centers, disps, index):
    R, t = icp(centers, disps[index])
    tmp = centers + disps[index]
    tmp = (R @ tmp.T).T + t

    return tmp-centers

In [ ]:
new_disps = []
for i in range(len(disps)):
    new_disps.append(align(centers, disps, i))

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='red')

vertices = np.vstack([centers, centers + new_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='linea', color='green')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='red')

    vertices = np.vstack([centers, centers + new_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='linea', color='green')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1), continuous_update=False)



